In [1]:
import os
import json
import uuid
from openai import OpenAI

In [4]:
class SyntheticGenerator:
    def __init__(self, api_key, model="gpt-4o-mini"):
        """
        初始化生成器
        :param api_key: 你的 OpenAI API Key
        :param model: 推荐使用具有强逻辑能力的模型
        """
        self.client = OpenAI(api_key=api_key)
        self.model = model

    def generate_full_package(self, topic):
        """
        核心方法：生成包含角色、分镜、交互点和推荐标签的完整数据包
        """
        system_prompt = """
        你是一个顶级的 AI 智能体导演，专门为 3-6 岁儿童设计互动动画绘本。
        你的任务是根据主题，输出一个高度结构化的 JSON 任务书。
        
        必须包含以下字段：
        1. story_id: 随机生成的唯一 ID。
        2. protagonist: 包含 name (名字) 和 visual_description (极其详细的视觉特征，如：穿着银色宇航服、红鼻子的蓝色圆脑袋机器人)。
        3. pages: 一个数组，每页包含：
           - scene_id: 页码。
           - narration: 给孩子读的旁白文本。
           - visual_prompt: 专门给 Stable Diffusion 用的图像描述，必须包含主角特征和当前动作。
           - motion_directive: 简单的动态指令（如：小猫在跳跃、星星在闪烁）。
           - interaction: 这一页结束后的互动问题（如：你猜小机器人会选哪扇门？）。
        4. rec_tags: 推荐系统用的标签（风格、情感、认知难度、主题）。
        """

        user_input = f"请围绕主题 '{topic}' 创作一个 3 页的互动绘本故事。"

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_input}
                ],
                response_format={ "type": "json_object" }
            )
            
            story_data = json.loads(response.choices[0].message.content)
            # 自动添加一些元数据
            story_data['meta_info'] = {
                "topic": topic,
                "version": "1.0",
                "generated_by": "DreamWeaver_SyntheticGenerator"
            }
            return story_data
        except Exception as e:
            print(f"生成失败: {e}")
            return None

    def save_to_file(self, data, filename=None):
        if not data:
            return
        if not filename:
            filename = f"story_{data.get('story_id', uuid.uuid4())}.json"
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
        print(f"✅ 故事数据包已成功保存至: {filename}")

In [3]:
# --- 运行测试 ---
if __name__ == "__main__":
    # 请确保已设置环境变量或在此处直接替换 KEY
    MY_API_KEY = os.getenv("OPENAI_API_KEY") 
    
    generator = SyntheticGenerator(api_key=MY_API_KEY)
    
    print("🚀 正在为你的初创项目生成第一个 AI 绘本数据包...")
    result = generator.generate_full_package("小机器人学习如何保护地球环境")
    
    if result:
        generator.save_to_file(result)
        # 打印一下主角设定，看看一致性基础打得好不好
        print(f"\n🌟 设定主角: {result['protagonist']['name']}")
        print(f"📝 视觉特征: {result['protagonist']['visual_description']}")

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable